# Portable datasets, timezone and metadata

Create a time series with explicit interpretation, organize it in a hierarchy,
inspect its metadata using ordinary JSON, and move it by copying the database.
The final section shows how to keep meaningful files under ordinary Git.

**Setup:** use a Jupyter Python kernel with this repository installed, as described
in the [README](../README.md#install). This notebook is independent of
[the time-series introduction](01_time_series.ipynb). Run cells in order.
It creates and cleans up its own temporary data. Git commands are optional manual
examples; the Python cells do not modify any Git repository.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import shutil
from jsonldb import FolderDB, jsonlfile

workspace = TemporaryDirectory(prefix="jsonldb-portability-")
source = Path(workspace.name) / "source"
source.mkdir()
# Choose precision before opening a new, empty database.
jsonlfile.save_jsonl_atomic(
    str(source / "config.meta"), {"config": {"timespec": "microseconds"}}
)
db = FolderDB(str(source), hierarchy_depth=2)
assert db.timespec == "microseconds"

## 1. Establish a table's interpretation before writing observations

Precision belongs to the database. Timezone belongs to the table, beside opaque
consumer metadata in a fixed-width first-line slot. Enabling/resizing slots can
rewrite existing tables; here the database is empty.

At depth 2, `prices.us.AAPL` lives at `prices/us/prices.us.AAPL.jsonl`. Its full
name stays in the filename. `UTC` normalizes to `+00:00`; an absent declaration
would mean unspecified, not inferred UTC.

In [ ]:
name = "prices.us.AAPL"
db.set_meta_slot_bytes(4096)
db.overwrite_dict(name, {})
db.set_timezone(name, "UTC")
k0 = datetime(2026, 1, 5, 9, 30, 0, 123456, tzinfo=timezone.utc)
k1 = k0 + timedelta(microseconds=1)
consumer_meta = {"source": "example-feed", "units": "USD"}
db.upsert_dict(name, {
    k0: {"price": 185.0}, k1: {"price": 185.1},
}, meta=consumer_meta)
assert db.read_timezone(name) == "+00:00"
assert db.get_file_list() == [name]
print("Discovered series:", db.search_file_list(r"^prices\.us\."))

## 2. Read metadata first and observations second

Matching aware keys are stored without an offset suffix. Automatic reads return
naive datetimes at the configured precision; callers obtain timezone separately.
`get_dict_with_meta` fixes metadata-before-rows order but is not a transaction.

In [ ]:
result = db.get_dict_with_meta(name)
assert result.meta == consumer_meta
assert list(result.rows) == [k0.replace(tzinfo=None), k1.replace(tzinfo=None)]
assert db.get_dict([name], k0, k0)[name] == {k0.replace(tzinfo=None): {"price": 185.0}}
print("Timezone:", db.read_timezone(name))
print("Consumer metadata:", result.meta)
print("Observations:", result.rows)

## 3. Keep metadata and time semantics separate

Clearing consumer data preserves timezone. A conflicting input offset is rejected,
not converted. Fixed offsets have no DST rules. Changing timezone on a populated
table requires a separate migration; the current setter refuses that operation.

In [ ]:
db.clear_meta(name)
assert db.read_meta(name) is None
assert db.read_timezone(name) == "+00:00"
db.upsert_dict(name, {}, meta=consumer_meta)
before = db.get_dict([name], auto_deserialize=False)[name]
conflicting = k0.replace(tzinfo=timezone(timedelta(hours=8)))
try:
    db.upsert_dict(name, {conflicting: {"price": 999.0}})
except ValueError as error:
    print("Expected offset refusal:", error)
else:
    raise AssertionError("A conflicting offset should be refused")
assert db.get_dict([name], auto_deserialize=False)[name] == before

## 4. Inspect the files without the library

Line one stores library properties and consumer data. Padding is valid JSON
whitespace. The following inspection uses only the standard library. The table
carries its timezone, but its precision still comes from `config.meta`.

In [ ]:
relative_table = Path("prices") / "us" / "prices.us.AAPL.jsonl"
table = source / relative_table
with table.open(encoding="utf-8") as stream:
    envelope = json.loads(stream.readline())["_meta"]
    observations = [json.loads(line) for line in stream if line.strip()]
assert envelope == {"v": 1, "timezone": "+00:00", "data": consumer_meta}
assert list(observations[0]) == ["2026-01-05T09:30:00.123456"]
assert len(observations) == 2
print("Envelope:", envelope)
print("Observation:", observations[0])
print("Configuration:", json.loads((source / "config.meta").read_text()))

## 5. Copy the database and rebuild a missing index

Stop writers while copying. Keep the observations, `config.meta` and `h.meta`;
retain quarantined data too if any exists. Indexes and statistics are rebuildable.
This example deliberately omits one index in the destination, then rebuilds
statistics and confirms that data and interpretation survived the move.

In [ ]:
destination = Path(workspace.name) / "copy"
shutil.copytree(source, destination)
copied_table = destination / relative_table
copied_index = Path(str(copied_table) + ".idx")
copied_index.unlink()
copied = FolderDB(str(destination))
copied.build_dbmeta()  # Recover indexes and refresh informational paths.
assert copied.timespec == "microseconds"
assert copied.hierarchy_depth == 2
assert copied.read_timezone(name) == "+00:00"
assert copied.read_meta(name) == consumer_meta
assert copied.get_dict([name], auto_deserialize=False)[name] == before
assert copied_table.read_bytes() == table.read_bytes()
assert copied_index.exists()
assert copied.get_dbmeta()[name]["path"] == str(copied_table)
# The copy is independent of later source changes.
db.upsert_dict(name, {k0: {"price": 200.0}})
assert copied.get_dict([name], k0, k0)[name][k0.replace(tzinfo=None)]["price"] == 185.0
print("Portable copy:", destination)

## 6. Optional Git snapshots

JSONLDB does not manage Git. For a repository rooted at your database directory,
track observations and settings, and usually ignore derived artifacts. The
following cell prepares an example `.gitignore` in the **temporary copy**.
It does not initialize a repository or make commits.

In [ ]:
(destination / ".gitignore").write_text("*.idx\n/db.meta\n/.jsonldb/\n", encoding="utf-8")
print("Example ignore rules:")
print((destination / ".gitignore").read_text())

For a real dataset, stop writes, change into its database directory, then run
ordinary Git commands when you want a snapshot:

```bash
git init
git add .gitignore config.meta h.meta prices/
git diff --cached
git commit -m "Snapshot observations and interpretation"
```

This command example assumes the hierarchy layout demonstrated above. For a flat
database, omit `h.meta` and add the intended table files. Include quarantined data
if present. Ignore patterns do not untrack files already committed. Git commits
need your normal local identity configuration.

Blank tombstones and compaction can add diff noise. Review changes before committing.
Resolve conflicting observations and configuration explicitly: a textually clean
merge is not proof of correct time-series semantics. Git does not coordinate writers.

## 7. Rebuild table indexes after a restore or external edit

A retained index can look fresh while referring to another version's bytes.
For the quiescent copy, discard visible table indexes and rebuild them. This
removes no observations. Preserve a backup before forced lint if you suspect
malformed data; lint can remove damage and records findings in `.jsonldb/lint.log`.

In [ ]:
for index in destination.rglob("*.jsonl.idx"):
    if not any(part.startswith(".") for part in index.relative_to(destination).parts):
        index.unlink()
restored = FolderDB(str(destination))
restored.build_dbmeta()
assert restored.get_dict([name], auto_deserialize=False)[name] == before
print("Restored data and index agree.")

## Cleanup and reference

A whole-folder copy carries configuration; copying one series requires compatible
precision and slot configuration at the destination. JSON text remains inspectable,
but legacy filenames and filesystem path limits can affect cross-platform moves.
No transaction or fsync-based power-loss durability is promised.

See [portability and Git](../docs/portability-and-git.md), [file format](../docs/file-format.md)
and [API reference](../docs/api.md) for the complete operating rules.

In [ ]:
workspace.cleanup()
assert not source.exists() and not destination.exists()
print("Temporary example data removed.")